# TP : Classification d'images Chats vs Chiens — MLP vs CNN

**Contexte** : La classification d'images est une application importante du Deep Learning et de la vision par ordinateur.
L'objectif est de développer deux modèles capables de distinguer automatiquement :
- les images contenant des chats (*Cats*),
- les images contenant des chiens (*Dogs*).

Deux architectures sont étudiées : **MLP** (Multi-Layer Perceptron) et **CNN** (Convolutional Neural Network), puis comparées.

## 1. Importation des bibliothèques

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

print("TensorFlow :", tf.__version__)
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

ModuleNotFoundError: No module named 'tensorflow'

## 2. Chargement du jeu de données

On utilise le jeu *cats_and_dogs_filtered* (2000 images d'entraînement, 1000 de validation).
Si vous avez votre propre dossier (`train/cats`, `train/dogs`, `validation/cats`, `validation/dogs`), modifiez simplement `BASE_DIR`.

In [ ]:
url = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"
zip_path = tf.keras.utils.get_file("cats_and_dogs_filtered.zip", origin=url, extract=True)
BASE_DIR = os.path.join(os.path.dirname(zip_path), "cats_and_dogs_filtered")

train_dir = os.path.join(BASE_DIR, "train")
val_dir   = os.path.join(BASE_DIR, "validation")

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", seed=SEED)
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", shuffle=False)

class_names = train_ds.class_names
print("Classes :", class_names)  # ['cats', 'dogs']

## 3. Exploration des données

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i][0])])
        plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Prétraitement et optimisation du pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

normalization = layers.Rescaling(1./255)   # pixels de [0,255] vers [0,1]

train_ds_n = train_ds.map(lambda x, y: (normalization(x), y)).cache().prefetch(AUTOTUNE)
val_ds_n   = val_ds.map(lambda x, y: (normalization(x), y)).cache().prefetch(AUTOTUNE)

## 5. Modèle 1 : Multi-Layer Perceptron (MLP)

Le MLP **aplatit** l'image en un vecteur (128×128×3 = 49 152 valeurs) puis applique des couches entièrement connectées.
Il ne tient pas compte de la structure spatiale de l'image.

In [ ]:
mlp = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    layers.Flatten(),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")
], name="MLP")

mlp.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
            loss="binary_crossentropy", metrics=["accuracy"])
mlp.summary()

In [ ]:
EPOCHS = 20
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

history_mlp = mlp.fit(train_ds_n, validation_data=val_ds_n,
                      epochs=EPOCHS, callbacks=[early_stop])

## 6. Modèle 2 : Convolutional Neural Network (CNN)

Le CNN utilise des **couches de convolution** et de **pooling** qui extraient des caractéristiques locales (bords, textures, formes)
tout en partageant les poids, ce qui réduit fortement le nombre de paramètres.

In [ ]:
cnn = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(256, activation="relu"),
    layers.Dense(1, activation="sigmoid")
], name="CNN")

cnn.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
            loss="binary_crossentropy", metrics=["accuracy"])
cnn.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

history_cnn = cnn.fit(train_ds_n, validation_data=val_ds_n,
                      epochs=EPOCHS, callbacks=[early_stop])

## 7. Courbes d'apprentissage

In [ ]:
def plot_history(history, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history.history["accuracy"], label="Train")
    ax[0].plot(history.history["val_accuracy"], label="Validation")
    ax[0].set_title(f"{title} - Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
    ax[1].plot(history.history["loss"], label="Train")
    ax[1].plot(history.history["val_loss"], label="Validation")
    ax[1].set_title(f"{title} - Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
    plt.show()

plot_history(history_mlp, "MLP")
plot_history(history_cnn, "CNN")

## 8. Évaluation : matrice de confusion et rapport de classification

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in val_ds_n]).ravel().astype(int)

def evaluate(model, name):
    y_prob = model.predict(val_ds_n, verbose=0).ravel()
    y_pred = (y_prob > 0.5).astype(int)
    print(f"===== {name} =====")
    print(classification_report(y_true, y_pred, target_names=class_names))
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues")
    plt.title(f"Matrice de confusion - {name}")
    plt.show()
    return model.evaluate(val_ds_n, verbose=0)

loss_mlp, acc_mlp = evaluate(mlp, "MLP")
loss_cnn, acc_cnn = evaluate(cnn, "CNN")

## 9. Étude comparative

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Modèle": ["MLP", "CNN"],
    "Paramètres": [mlp.count_params(), cnn.count_params()],
    "Accuracy (val)": [round(acc_mlp, 4), round(acc_cnn, 4)],
    "Loss (val)": [round(loss_mlp, 4), round(loss_cnn, 4)],
    "Accuracy (train)": [round(history_mlp.history["accuracy"][-1], 4),
                         round(history_cnn.history["accuracy"][-1], 4)],
})
display(comparison)

plt.figure(figsize=(7, 4))
plt.plot(history_mlp.history["val_accuracy"], label="MLP")
plt.plot(history_cnn.history["val_accuracy"], label="CNN")
plt.title("Accuracy de validation : MLP vs CNN")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.grid(True)
plt.show()

## 10. Prédictions sur des exemples

In [ ]:
plt.figure(figsize=(12, 8))
for images, labels in val_ds.take(1):
    imgs_n = normalization(images)
    p_mlp = mlp.predict(imgs_n, verbose=0).ravel()
    p_cnn = cnn.predict(imgs_n, verbose=0).ravel()
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f"Réel: {class_names[int(labels[i][0])]}\n"
                  f"MLP: {class_names[int(p_mlp[i] > .5)]} | CNN: {class_names[int(p_cnn[i] > .5)]}",
                  fontsize=9)
        plt.axis("off")
plt.tight_layout()
plt.show()

## 11. Conclusion

| Critère | MLP | CNN |
|---|---|---|
| Entrée | Image aplatie (perd la structure spatiale) | Image 2D conservée |
| Extraction de caractéristiques | Aucune (apprend directement sur les pixels) | Automatique (filtres de convolution) |
| Nombre de paramètres | Très élevé dans la 1ʳᵉ couche dense | Réduit grâce au partage des poids |
| Invariance à la translation | Non | Oui (convolution + pooling) |
| Risque de surapprentissage | Élevé | Plus faible |
| Performance attendue | ~55–65 % | ~75–85 % (sans augmentation de données) |

**Conclusion** : le CNN est nettement mieux adapté à la classification d'images : il exploite la structure spatiale, nécessite moins de paramètres
et généralise mieux que le MLP.

**Pistes d'amélioration** : data augmentation (`RandomFlip`, `RandomRotation`, `RandomZoom`), *Batch Normalization*, transfer learning (MobileNetV2, VGG16).